# 05 — Final Load Prep

Validates all processed exports and confirms they are ready for Tableau Public ingestion.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
PROC = PROJECT_ROOT / 'data' / 'processed'

In [2]:
exports = {
    'olist_master_dataset'     : PROC / 'olist_master_dataset.csv',
    'olist_geospatial_summary' : PROC / 'olist_geospatial_summary.csv',
    'olist_rfm_segments'       : PROC / 'olist_rfm_segments.csv',
    'olist_rfm_summary'        : PROC / 'olist_rfm_summary.csv',
}

for name, path in exports.items():
    df = pd.read_csv(path)
    print(f'{name:30s}: {df.shape}  |  nulls={df.isnull().sum().sum():,}')

olist_master_dataset          : (98207, 36)  |  nulls=10,989
olist_geospatial_summary      : (27, 8)  |  nulls=0
olist_rfm_segments            : (94990, 10)  |  nulls=0
olist_rfm_summary             : (7, 5)  |  nulls=0


In [3]:
master = pd.read_csv(PROC / 'olist_master_dataset.csv',
                     parse_dates=['order_purchase_timestamp'])

print(f'Date range       : {master["order_purchase_timestamp"].min().date()} → {master["order_purchase_timestamp"].max().date()}')
print(f'Total orders     : {len(master):,}')
print(f'Unique customers : {master["customer_unique_id"].nunique():,}')
print(f'Order statuses   : {master["order_status"].value_counts().to_dict()}')
print(f'On-time rate     : {(~master["is_late"].fillna(True)).sum() / master["is_late"].notna().sum():.2%}')
print(f'Total revenue    : R${master["revenue"].sum():,.2f}')

Date range       : 2016-09-04 → 2018-09-03
Total orders     : 98,207
Unique customers : 94,990
Order statuses   : {'delivered': 96478, 'shipped': 1107, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}
On-time rate     : 93.35%
Total revenue    : R$13,494,400.74


In [4]:
master.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,seller_zip_code_prefix,seller_city,seller_state,review_score,review_comment_message,delivery_time_days,delivery_delay_days,is_late,order_year_month,order_year
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,9350.0,maua,SP,4.0,"Não testei o produto ainda, mas ele veio corre...",8.0,-8.0,False,2017-10,2017
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,31570.0,belo horizonte,SP,4.0,Muito bom o produto.,13.0,-6.0,False,2018-07,2018
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,14840.0,guariba,SP,5.0,No Comment,9.0,-18.0,False,2018-08,2018
